# Brain Tumor MRI Experiments

This notebook provides an end-to-end experimental workflow for the brain tumor classification project.

Sections:
1. Setup & Imports
2. Dataset Exploration
3. Preprocessing Visualization
4. Data Loader Test
5. Model Architecture Summary
6. Training / Evaluate-only Run
7. Results Comparison
8. Error Analysis

## 1) Setup & Imports

Initialize environment, append project root to `sys.path`, import project modules, and load configuration.

In [ ]:
# 1) Setup & Imports
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from torchvision.utils import make_grid

def _resolve_project_root() -> Path:
    """Find repo root (folder with main.py + src/) for Colab and local runs."""
    env = os.environ.get("BRAIN_TUMOR_CLASSIFIER_ROOT")
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "main.py").is_file() and (p / "src").is_dir():
            return p
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "main.py").is_file() and (p / "src").is_dir():
            return p
    raise RuntimeError(
        "Could not find project root. In Colab: %cd /content/brain_tumor_classifier "
        "or set os.environ['BRAIN_TUMOR_CLASSIFIER_ROOT'] to that path."
    )


PROJECT_ROOT = _resolve_project_root()
root_s = str(PROJECT_ROOT)
if root_s not in sys.path:
    sys.path.insert(0, root_s)

from src.preprocessing import PreprocessingPipeline, enhance_image, get_train_transforms, remove_noise
from src.utils import get_data_loaders
from src.models import get_model

# Load config
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_cfg = config["data"]
model_cfg = config["model"]
paths_cfg = config["paths"]

RAW_DIR = PROJECT_ROOT / data_cfg["raw_dir"]
PROCESSED_DIR = PROJECT_ROOT / data_cfg["processed_dir"]
PLOTS_DIR = PROJECT_ROOT / paths_cfg["plots"]
CHECKPOINTS_DIR = PROJECT_ROOT / paths_cfg["checkpoints"]

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)
print("Config loaded successfully.")
config

In [ ]:
## 2) Dataset Exploration

This section inspects raw class balance and visualizes sample MRI scans before preprocessing.

In [ ]:
# Count images per class in data/raw
valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
class_counts = {}
class_to_paths = {}

for class_dir in sorted([p for p in RAW_DIR.iterdir() if p.is_dir()]):
    paths = [
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in valid_ext
    ]
    class_counts[class_dir.name] = len(paths)
    class_to_paths[class_dir.name] = sorted(paths)

counts_df = pd.DataFrame({"class": list(class_counts.keys()), "count": list(class_counts.values())})
counts_df

In [ ]:
# Plot class distribution bar chart
plt.figure(figsize=(8, 5))
plt.bar(counts_df["class"], counts_df["count"], color="steelblue")
plt.title("Raw Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Image Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Show sample MRI images (2 per class)
num_classes = len(class_to_paths)
fig, axes = plt.subplots(num_classes, 2, figsize=(8, 3 * max(1, num_classes)))
if num_classes == 1:
    axes = np.array([axes])

for row, (class_name, paths) in enumerate(class_to_paths.items()):
    for col in range(2):
        ax = axes[row, col]
        if col < len(paths):
            img = Image.open(paths[col]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"{class_name} #{col + 1}")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 3) Preprocessing Visualization

This section shows a visual walkthrough of preprocessing: original image, denoised image, enhanced image, and augmented output.

In [ ]:
# Take one sample image and visualize preprocessing stages
first_class = next(iter(class_to_paths))
sample_path = class_to_paths[first_class][0]

orig = np.array(Image.open(sample_path).convert("RGB"))
denoised = remove_noise(orig, method="gaussian")
enhanced = enhance_image(denoised)
aug_transform = get_train_transforms(img_size=data_cfg.get("img_size", 224))
augmented_tensor = aug_transform(image=enhanced)["image"]

# De-normalize for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
augmented = augmented_tensor.detach().cpu() * std + mean
augmented = torch.clamp(augmented, 0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(orig)
axes[0].set_title("Original")
axes[1].imshow(denoised)
axes[1].set_title("Denoised")
axes[2].imshow(enhanced)
axes[2].set_title("Enhanced")
axes[3].imshow(augmented)
axes[3].set_title("Augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4) Data Loader Test

Instantiate train/validation/test loaders, verify split sizes, and visualize one training batch.

In [ ]:
# Build dataloaders from processed data
if not PROCESSED_DIR.exists() or len(list(PROCESSED_DIR.rglob("*"))) == 0:
    print("Processed data not found. Running preprocessing now...")
    pipeline = PreprocessingPipeline(config=data_cfg)
    pipeline.process_dataset(str(RAW_DIR), str(PROCESSED_DIR))

train_loader, val_loader, test_loader, class_names = get_data_loaders(str(PROCESSED_DIR), config)

print("Class names:", class_names)
print("Train samples:", len(train_loader.dataset))
print("Val samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

In [ ]:
# Show one batch as a grid
images, labels = next(iter(train_loader))
inv_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
inv_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
vis_images = torch.clamp(images[:16].cpu() * inv_std + inv_mean, 0, 1)

grid = make_grid(vis_images, nrow=4, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("Sample Training Batch (first 16 images)")
plt.axis("off")
plt.show()

print("First 16 labels:", [class_names[i] for i in labels[:16].tolist()])

## 5) Model Architecture Summary

Print architecture summaries for ResNet50, ViT, and Hybrid models using `torchsummary` (with a fallback to plain model printout).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

try:
    from torchsummary import summary
    has_torchsummary = True
except ImportError:
    has_torchsummary = False
    print("torchsummary is not installed. Install with: pip install torchsummary")

for model_name in ["resnet50", "vit", "hybrid"]:
    print("\n" + "=" * 80)
    print(f"Model: {model_name}")
    model = get_model(model_name, num_classes=model_cfg.get("num_classes", 4), pretrained=False).to(device)
    if has_torchsummary:
        summary(model, input_size=(3, 224, 224), device=device)
    else:
        print(model)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total_params:,}")

## 6) Training Cell (Optional)

You can either run the full training script or load existing results from `results/final_results.csv`.

In [ ]:
# Option A: Run full pipeline from notebook (uncomment to execute)
# %cd ..
# !python main.py --config configs/config.yaml

# Option B: Load pre-saved results
final_results_path = PROJECT_ROOT / "results" / "final_results.csv"
if final_results_path.exists():
    final_results_df = pd.read_csv(final_results_path)
    print("Loaded existing results from:", final_results_path)
    display(final_results_df)
else:
    print("No pre-saved results found at:", final_results_path)
    print("Run main.py first or save results manually.")

## 7) Results Comparison

Load final model results and visualize metric comparisons.

In [ ]:
results_csv = PROJECT_ROOT / "results" / "final_results.csv"
if not results_csv.exists():
    raise FileNotFoundError(f"Results CSV not found: {results_csv}")

results_df = pd.read_csv(results_csv)
print("Results table:")
display(results_df)

plot_df = results_df.copy()
if "model_name" in plot_df.columns:
    plot_df = plot_df.set_index("model_name")

metric_cols = [c for c in ["accuracy", "precision", "recall", "specificity", "f1_score", "auc_roc"] if c in plot_df.columns]

ax = plot_df[metric_cols].plot(kind="bar", figsize=(10, 5), rot=20)
ax.set_title("Model Comparison Across Metrics")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8) Error Analysis

Load the best checkpoint, run inference on the test set, and inspect misclassified examples with true vs predicted labels.

In [ ]:
# Choose model for error analysis
analyze_model_name = "hybrid"  # change to: resnet50 / vit / hybrid
checkpoint_path = CHECKPOINTS_DIR / f"{analyze_model_name}_best.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = get_model(analyze_model_name, num_classes=len(class_names), pretrained=False).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

misclassified = []
max_show = 16

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        mismatch_idx = torch.where(preds != labels)[0].tolist()
        for idx in mismatch_idx:
            misclassified.append(
                {
                    "image": images[idx].detach().cpu(),
                    "true": int(labels[idx].item()),
                    "pred": int(preds[idx].item()),
                }
            )
            if len(misclassified) >= max_show:
                break
        if len(misclassified) >= max_show:
            break

print(f"Misclassified samples collected: {len(misclassified)}")

if len(misclassified) == 0:
    print("No misclassifications found in inspected batches.")
else:
    ncols = 4
    nrows = int(np.ceil(len(misclassified) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = np.array(axes).reshape(nrows, ncols)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    for i, item in enumerate(misclassified):
        r, c = divmod(i, ncols)
        img = torch.clamp(item["image"] * std + mean, 0, 1).permute(1, 2, 0).numpy()
        axes[r, c].imshow(img)
        axes[r, c].set_title(
            f"True: {class_names[item['true']]}\nPred: {class_names[item['pred']]}",
            fontsize=9,
        )
        axes[r, c].axis("off")

    for j in range(len(misclassified), nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.suptitle(f"Misclassified Samples ({analyze_model_name})", y=1.02)
    plt.tight_layout()
    plt.show()